# Effect of the Thomson scattering calibration uncertainty, against the interferometer

How the calibration uncertainty changes the inferred profiles and bands of the eight
ground-truth plasma states, and how many interferometer chords it takes before it stops
mattering. Each scattering volume has a correction `delta_j` (`C_j = C_j^geom * delta_j`)
with the prior `ln delta_j ~ U(-ln N, ln N)`, swept over the bound `N`; `N = 1` infers no
`delta_j` and is the reference. The second axis is the chord configuration, 0 to 5
horizontal chords at chosen heights between the central chord and the top one
(`run_study.CHORD_FRACTIONS`). Every cell's placement carries the union fan of all
configurations and masks the chords it does not read, so a chord at a given height
carries the same noisy reading in every configuration; the data always have
`delta_j = 1`, so every cell of a state fits the same Thomson signals.

A run counts as converged here when the chi2 per degree of freedom of its reported MAP
is below `CHI2_PER_DOF_LIMIT`. Every run is kept and listed; the statistics and figures
read the converged ones.

```
python src/plasma_state_estimation_for_diagnostic_placement_optimization/results_plots_tables/results/plasma_state_estimation/ts_uncertainty_level_vs_inference/run_study.py --workers 5
```

The runner stores the runs and writes `tables/`; this notebook writes `figures/`.

In [ ]:
import functools
import pathlib

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
import numpyro
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.bayesian_inference import (  # noqa: E501
    desc_truth_types,
    inference_model_utils,
    inference_target_types,
    laplace_approximation_utils,
    priors_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    geometry_utils,
    thomson_scattering_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_placement_optimization import (  # noqa: E501
    diagnostics_placement_optimization_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
    model_storage_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.diagnostics_placement_optimization.if_and_ts_placement_study import (  # noqa: E501
    run_scan as joint_placement_study,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.diagnostics_placement_optimization.if_placement_study import (  # noqa: E501
    run_study as interferometer_placement_study,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    inference_metrics_utils,
    inference_store_utils,
    plots_diagnostics_forward_models,
    profile_table_utils,
    pubfig,
    sweep_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_plasma_state_estimation as plots,
)

pubfig.apply_style()
pd.set_option("display.max_colwidth", None)

PACKAGE_DIRECTORY = pathlib.Path(plots.__file__).parents[2]
NOTEBOOK_DIRECTORY = (
    PACKAGE_DIRECTORY / "results_plots_tables" / "results" / "plasma_state_estimation"
) / "ts_uncertainty_level_vs_inference"
FIGURES_DIRECTORY = NOTEBOOK_DIRECTORY / "figures"
DATA_DIRECTORY = PACKAGE_DIRECTORY.parents[1] / "data"
PARQUET = DATA_DIRECTORY / f"{dataset_utils.DEFAULT_DATASET}.parquet"
SETTINGS = DATA_DIRECTORY / f"{dataset_utils.DEFAULT_DATASET}_settings.json"

CHI2_PER_DOF_LIMIT = 3.0
"""A run whose reported MAP has chi2 / dof above this is a local minimum: kept and
listed, but left out of every statistic and figure below."""

# The current run of every state, level, chord configuration and noise draw, as the
# runner's manifest lists.
records = [
    inference_store_utils.load_record(inference_store_utils.INFERENCE_STORE, identifier)
    for identifier in sweep_utils.current_ids(
        NOTEBOOK_DIRECTORY / "inference_ids.json"
    ).values()
]
truths = sorted({record.truth_name for record in records})
levels = sorted({record.settings.thomson_calibration_bound for record in records})


def chord_count(record):
    """How many chords the run read; its placement carries the whole fan."""
    return len(record.interferometer_chord_indices or ())


def converged(record):
    """Whether the reported MAP fits the data, as this notebook counts convergence."""
    return record.chi_squared / record.n_degrees_of_freedom < CHI2_PER_DOF_LIMIT


chord_counts = sorted({chord_count(record) for record in records})
converged_records = [record for record in records if converged(record)]
failed = pd.DataFrame(
    [
        {
            "truth": record.truth_name,
            "chords": chord_count(record),
            "N": record.settings.thomson_calibration_bound,
            "chi2 / dof": record.chi_squared / record.n_degrees_of_freedom,
        }
        for record in records
        if not converged(record)
    ]
)
print(
    f"{len(records)} runs over {len(truths)} states, {len(levels)} levels of N and "
    f"{len(chord_counts)} chord configurations; {len(failed)} not converged "
    f"(chi2 / dof above {CHI2_PER_DOF_LIMIT:g}), left out of every statistic below"
)
if len(failed):
    display(
        failed.pivot_table(
            index="chords", columns="N", values="truth", aggfunc="count", fill_value=0
        ).rename_axis(columns="runs not converged, N")
    )

In [ ]:
record = records[0]
placement = inference_store_utils.load_placement(
    inference_store_utils.PLACEMENT_STORE, record.diagnostics_placement_id
)
seeds = sorted({record.settings.seed for record in records})
draws = len(records) // (len(truths) * len(levels) * len(chord_counts))
pd.Series(
    {
        "ground-truth plasma states": ", ".join(truths),
        "noise draws per state": f"{draws} (seeds {seeds[0]} to {seeds[-1]})",
        "calibration bounds N": ", ".join(f"{level:g}" for level in levels),
        "chord configurations": ", ".join(f"{count}" for count in chord_counts),
        "chords in the fan": len(placement.interferometer_chords),
        "Thomson noise model": placement.design.thomson_noise_model,
        "Thomson scattering volumes": len(placement.thomson_geometry.volume_positions),
        "interferometer sigma_IF [m^-2]": f"{record.settings.interferometer_sigma:.3g}",
        "profile family": "two-power, center * (1 - rho^alpha)^beta",
        "MAP starts": f"{record.settings.map.n_starts}, drawn from the priors",
        "posterior": "Laplace Gaussian at the best converged MAP start",
        "draws per band": record.settings.n_posterior_samples,
    },
    name="setting",
).to_frame()

## Chord configurations

One panel per configuration with at least one chord, with the chords the likelihood reads on the reference
equilibrium the placement was built on. The configurations are not nested: each spreads
its chords evenly between the central chord and the top one.

In [ ]:
reference = desc_truth_types.read_desc_truth(
    desc_truth_types.REFERENCE_EQUILIBRIUM_ID, PARQUET, SETTINGS
)
indices_of = {
    chord_count(record): tuple(record.interferometer_chord_indices or ())
    for record in records
}
# The configuration without chords would draw an empty cross-section, so it is left out.
drawn_counts = [count for count in chord_counts if count > 0]
figure, axes_row = pubfig.subplots(1, len(drawn_counts), width="double", aspect=2.6)
for axes, count in zip(axes_row, drawn_counts):
    plots_diagnostics_forward_models.draw_interferometer_layout(
        axes,
        reference.coefficients,
        [placement.interferometer_chords[index] for index in indices_of[count]],
        placement.design,
    )
    axes.set_title(rf"$N_{{\mathrm{{LoS}}}} = {count}$")
for axes in axes_row[1:]:
    axes.set_ylabel("")
    axes.tick_params(labelleft=False)
figure.legend(
    handles=axes_row[0].get_legend_handles_labels()[0],
    ncol=3,
    loc="outside lower center",
)
pubfig.save(figure, str(FIGURES_DIRECTORY / "ts_calibration_bound_chord_layouts"))
display(figure)
plt.close(figure)

## 1. Did the MAP converge?

Same optimizer and figure as in `mcmc_benchmark`, one convergence figure per chord
configuration with the quantiles over the states and the levels of `N`. The table lists
every run: `map converged` is the optimizer's own verdict (Newton decrement at a
positive-definite curvature), `converged` the chi2 test of this notebook, which also
catches a certified local minimum.

In [ ]:
map_table = pd.DataFrame(
    [
        {
            "truth": record.truth_name,
            "chords": chord_count(record),
            "N": record.settings.thomson_calibration_bound,
            "map converged": record.map_converged,
            "|grad U|_inf at MAP": record.map_gradient_inf_norm,
            "chi2 / dof": record.chi_squared / record.n_degrees_of_freedom,
            "converged": converged(record),
        }
        for record in records
    ]
).sort_values(["truth", "chords", "N"], ignore_index=True)
display(map_table.round({"|grad U|_inf at MAP": 4, "chi2 / dof": 2}))

truth_profiles = {
    truth: desc_truth_types.read_desc_truth(truth, PARQUET, SETTINGS).profiles
    for truth in truths
}
"""# One start per run, so the quantiles run over the states and the levels of each
# configuration.
for count in chord_counts:
    figure = plots.plot_map_convergence(
        [
            (
                record,
                inference_store_utils.load_inference(
                    inference_store_utils.INFERENCE_STORE, record.inference_id
                )[1],
                truth_profiles[record.truth_name],
            )
            for record in sorted(
                records,
                key=lambda r: (r.truth_name, r.settings.thomson_calibration_bound),
            )
            if chord_count(record) == count
        ],
        label=f"{count} chords",
    )
    pubfig.save(
        figure, str(FIGURES_DIRECTORY / f"ts_calibration_bound_convergence_if{count}")
    )
    display(figure)
    plt.close(figure)"""

## 2. Effect on the inferred profiles

One figure per state, level and chord configuration, drawn as in `mcmc_benchmark`, with
the prior draw the MAP started from. The Thomson points of a state are the same in every
cell; only the chords read, the band and the MAP change.

In [ ]:
# Every cell of a state shares its signals, so its Thomson points are rebuilt once.
thomson_points = {
    truth: profile_table_utils.thomson_points_table(
        next(r for r in records if r.truth_name == truth),
        desc_truth_types.read_desc_truth(truth, PARQUET, SETTINGS),
    )
    for truth in truths
}
"""for record in sorted(
    records,
    key=lambda r: (r.truth_name, chord_count(r), r.settings.thomson_calibration_bound),
):
    level = record.settings.thomson_calibration_bound
    count = chord_count(record)
    figure = plots.plot_method_band_comparison(
        profile_table_utils.profile_band_table(record, "map_laplace"),
        ("map_laplace",),
        thomson_points=thomson_points[record.truth_name],
        title=rf"{record.truth_name}, $f_\delta$ = {level:g}, {count} chords",
    )
    # A dot in the stem would read as a file suffix, so the point becomes `p`.
    name = (
        f"ts_calibration_bound_profiles_{record.truth_name}_n{level:g}_if{count}"
    ).replace(".", "p")
    pubfig.save(figure, str(FIGURES_DIRECTORY / name))
    display(figure)
    plt.close(figure)"""

## 3. Effect on the uncertainty and the error

RMSE and band half-width of `ne` and `Te` against `N` and the chord configuration, over
the converged runs of the states: first as heat maps (`N` across, chords up, the number
in a cell is the mean the colour encodes), then as lines against `N`, one per chord
configuration, and per configuration with every state's own curve behind the mean. The
x axis of the line figures is `ln N`, the half-width of the prior box. The two RMSE
panels share one scale and the two band panels another, the band on a log axis. The
table gives each configuration's mean as a multiple of its value at `N = 1`.

In [ ]:
metrics = inference_metrics_utils.sweep_metrics_table(converged_records)
level_of = {r.inference_id: r.settings.thomson_calibration_bound for r in records}
metrics["N"] = metrics["inference_id"].map(level_of)
metrics = metrics.loc[
    metrics["name"].isin(["electron_density", "electron_temperature"])
]
metrics["profile"] = metrics["name"].map(
    {"electron_density": "ne", "electron_temperature": "Te"}
)
metrics = metrics.rename(columns={"n_interferometer_chords": "chords"})

PROFILES = (("ne", r"$n_e$"), ("Te", r"$T_e$"))
ROWS = (
    ("error_percent", "RMSE [% of peak]"),
    ("uncertainty_percent", "band half-width [% of value]"),
)

In [ ]:
def metric_grid(metrics, column, profile, chord_counts, levels):
    """Mean of one metric per chord configuration (rows) and level of N (columns)."""
    return (
        metrics[metrics["profile"] == profile]
        .pivot_table(index="chords", columns="N", values=column)
        .reindex(index=chord_counts, columns=levels)
    )


colormap = plt.get_cmap("viridis").copy()
colormap.set_bad("0.9")
figure, axes_grid = pubfig.subplots(2, 2, width="double", aspect=0.7)
for row_axes, (column, label) in zip(axes_grid, ROWS):
    grids = [
        np.asarray(metric_grid(metrics, column, profile, chord_counts, levels))
        for profile, _ in PROFILES
    ]
    # One colour scale per quantity, shared by its ne and Te panels.
    finite = np.concatenate([grid[np.isfinite(grid)] for grid in grids])
    low, high = finite.min(), finite.max()
    midpoint = 0.5 * (low + high)
    for axes, values, (_, title) in zip(row_axes, grids, PROFILES):
        n_rows, n_columns = values.shape
        mesh = axes.pcolormesh(
            np.arange(n_columns + 1) - 0.5,
            np.arange(n_rows + 1) - 0.5,
            np.ma.masked_invalid(values),
            cmap=colormap,
            vmin=low,
            vmax=high,
        )
        # White text on the dark half of the colour scale, black on the light half.
        for row in range(n_rows):
            for col in range(n_columns):
                if np.isfinite(values[row, col]):
                    axes.text(
                        col,
                        row,
                        f"{values[row, col]:.1f}",
                        ha="center",
                        va="center",
                        fontsize="x-small",
                        color="w" if values[row, col] < midpoint else "k",
                    )
        axes.set_xticks(np.arange(n_columns), [f"{level:g}" for level in levels])
        axes.set_yticks(np.arange(n_rows), [str(count) for count in chord_counts])
        axes.set_title(title)
    figure.colorbar(mesh, ax=list(row_axes), label=label, shrink=0.9)
for axes in axes_grid[1]:
    axes.set_xlabel(r"$f_\delta$")
for axes in axes_grid[:, 0]:
    axes.set_ylabel("interferometer chords")
pubfig.save(figure, str(FIGURES_DIRECTORY / "ts_calibration_bound_heatmap"))
display(figure)
plt.close(figure)

In [ ]:
"""PANELS = tuple(
    (profile, column, f"{title} {label}")
    for column, label in ROWS
    for profile, title in PROFILES
)
# The prior half-width is ln N, so that is the axis; the levels label its ticks, 1.05
# on a second line since 1, 1.05 and 1.2 sit within 0.18 of each other.
positions = np.log(levels)
tick_labels = [f"\n{level:g}" if level == 1.05 else f"{level:g}" for level in levels]


def summary_axes(axes, label, column, positions, tick_labels):
    axes.set_ylabel(label)
    # The band spans more than a decade across the configurations; the RMSE does not.
    if column == "uncertainty_percent":
        axes.set_yscale("log")
        axes.yaxis.set_major_locator(
            matplotlib.ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0))
        )
        axes.yaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter("%g"))
        axes.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    else:
        axes.set_ylim(0.0, None)
    axes.set_xticks(positions, labels=tick_labels)
    axes.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())


# The configurations are ordered, so a sequential colormap reads as their order.
colours = plt.cm.viridis(np.linspace(0.0, 0.85, len(chord_counts)))
figure, axes_grid = pubfig.subplots(
    2, 2, width="double", aspect=0.8, sharex=True, sharey="row"
)
for axes, (profile, column, label) in zip(axes_grid.ravel(), PANELS):
    rows = metrics[metrics["profile"] == profile]
    for colour, count in zip(colours, chord_counts):
        mean = rows[rows["chords"] == count].groupby("N")[column].mean().reindex(levels)
        axes.plot(
            positions, mean, "o-", color=colour, lw=1.4, ms=3.5, label=f"{count} chords"
        )
    summary_axes(axes, label, column, positions, tick_labels)
for axes in axes_grid[1]:
    axes.set_xlabel(r"$f_\delta$")
axes_grid[0, 0].legend(frameon=False, loc="upper left", fontsize="small")
pubfig.save(figure, str(FIGURES_DIRECTORY / "ts_calibration_bound_summary"))
display(figure)
plt.close(figure)

for count in chord_counts:
    figure, axes_grid = pubfig.subplots(
        2, 2, width="double", aspect=0.8, sharex=True, sharey="row"
    )
    for axes, (profile, column, label) in zip(axes_grid.ravel(), PANELS):
        rows = metrics[(metrics["profile"] == profile) & (metrics["chords"] == count)]
        for truth in truths:
            state = rows[rows["truth_name"] == truth].sort_values("N")
            axes.plot(
                np.log(state["N"]),
                state[column],
                "o-",
                color=pubfig.PALETTE[5],
                lw=0.8,
                ms=2.5,
                alpha=0.55,
            )
        mean = rows.groupby("N")[column].mean().reindex(levels)
        axes.plot(positions, mean, "o-", color=pubfig.PALETTE[1], lw=1.6, ms=4.0)
        summary_axes(axes, label, column, positions, tick_labels)
    for axes in axes_grid[1]:
        axes.set_xlabel(r"$f_\delta$")
    axes_grid[0, 0].plot(
        [],
        [],
        "o-",
        color=pubfig.PALETTE[5],
        lw=0.8,
        ms=2.5,
        alpha=0.55,
        label="single state",
    )
    axes_grid[0, 0].plot(
        [],
        [],
        "o-",
        color=pubfig.PALETTE[1],
        lw=1.6,
        ms=4.0,
        label="mean of the states",
    )
    axes_grid[0, 0].legend(frameon=False, loc="upper left", title=f"{count} chords")
    pubfig.save(
        figure, str(FIGURES_DIRECTORY / f"ts_calibration_bound_summary_if{count}")
    )
    display(figure)
    plt.close(figure)

relative = metrics.groupby(["chords", "profile", "N"])[
    ["uncertainty_percent", "error_percent"]
].mean()
relative = relative / relative.xs(levels[0], level="N")
relative.columns = ["band half-width", "RMSE"]
relative.unstack("profile").reindex(columns=["ne", "Te"], level=1).round(3)"""

## 4. Recovered calibration corrections

How much of the prior on each `delta_j` the data remove at each `N > 1` and chord
configuration, `(Delta_prior - Delta_posterior) / Delta_prior` with `Delta_posterior`
the width in `ln delta_j` of the 5 to 95% interval of the states' Laplace bands, pooled
as an equal-weight mixture, and `Delta_prior` the prior's, `0.9 * 2 ln N`. Zero means
the data left the prior unchanged, one means they fixed `delta_j`. The heat map is the
mean over the fifteen volumes, `N` across and chords up as in section 3; the `N = 1`
column is grey since no correction is inferred there. The table has every volume,
volume 0 being the outboard edge. Converged runs only.

In [ ]:
calibration = pd.concat(
    [
        profile_table_utils.calibration_table(
            record,
            inference_store_utils.load_inference(
                inference_store_utils.INFERENCE_STORE, record.inference_id
            )[1],
        ).assign(chords=chord_count(record))
        for record in converged_records
        if record.settings.thomson_calibration_bound > 1.0
    ],
    ignore_index=True,
)
volume_positions = np.asarray(placement.thomson_geometry.volume_positions)
major_radius = np.hypot(volume_positions[:, 0], volume_positions[:, 1])
z_95 = float(jax.scipy.stats.norm.ppf(0.95))
pooled_rows = []
for (count, bound, volume), rows in calibration.groupby(
    ["chords", "bound", "volume_index"]
):
    prior = next(iter(priors_utils.calibration_priors([0], bound).values()))
    transform = numpyro.distributions.transforms.biject_to(prior.support)
    # Each band is a Gaussian in the unconstrained coordinate, recovered from its
    # quantiles. Their equal-weight mixture holds each state's own width and the
    # scatter between the states, which a mean of the widths would drop.
    lower = np.asarray(transform.inv(jnp.asarray(rows["q05"].to_numpy())))
    upper = np.asarray(transform.inv(jnp.asarray(rows["q95"].to_numpy())))
    mean, sigma = 0.5 * (lower + upper), (upper - lower) / (2.0 * z_95)
    grid = np.linspace((mean - 8.0 * sigma).min(), (mean + 8.0 * sigma).max(), 20001)
    cdf = np.mean(
        np.asarray(jax.scipy.stats.norm.cdf(grid[:, None], mean, sigma)), axis=1
    )
    q05, q95 = np.asarray(transform(jnp.asarray(np.interp([0.05, 0.95], cdf, grid))))
    pooled_rows.append(
        {
            "chords": count,
            "N": bound,
            "volume": volume,
            "R [m]": major_radius[volume],
            "retained width": np.log(q95 / q05) / (0.9 * 2.0 * np.log(bound)),
        }
    )
pooled = pd.DataFrame(pooled_rows)
# The share of the prior the data removed, so a well-determined correction reads
# as one and an untouched prior as zero.
pooled["removed width"] = 1.0 - pooled["retained width"]

removed = (
    pooled.groupby(["chords", "N"])["removed width"]
    .mean()
    .unstack("N")
    .reindex(index=chord_counts, columns=levels)
)
values = np.asarray(removed, dtype=float)
n_rows, n_columns = values.shape
# No correction is inferred at N = 1, so its column has no value and stays grey.
colormap = plt.get_cmap("viridis").copy()
colormap.set_bad("0.9")
figure, (axes,) = pubfig.subplots(width="single", aspect=0.8)
# A fraction of the prior, so the scale is 0 to 1 whatever the data remove.
mesh = axes.pcolormesh(
    np.arange(n_columns + 1) - 0.5,
    np.arange(n_rows + 1) - 0.5,
    np.ma.masked_invalid(values),
    cmap=colormap,
    vmin=0.0,
    vmax=1.0,
)
figure.colorbar(
    mesh,
    ax=axes,
    label=(
        r"$(\Delta_\mathrm{prior} - \Delta_\mathrm{posterior})"
        r" / \Delta_\mathrm{prior}$"
    ),
    shrink=0.9,
)
for row in range(n_rows):
    for col in range(n_columns):
        if np.isfinite(values[row, col]):
            axes.text(
                col,
                row,
                f"{values[row, col]:.2f}",
                ha="center",
                va="center",
                fontsize="x-small",
                color="w" if values[row, col] < 0.5 else "k",
            )
axes.set_xticks(np.arange(n_columns), [f"{level:g}" for level in levels])
axes.set_yticks(np.arange(n_rows), [str(count) for count in chord_counts])
axes.set_xlabel(r"$f_\delta$")
axes.set_ylabel("interferometer chords")
pubfig.save(figure, str(FIGURES_DIRECTORY / "ts_calibration_bound_corrections"))
display(figure)
plt.close(figure)

pooled.pivot(
    index=["chords", "volume", "R [m]"], columns="N", values="removed width"
).round(3)

Per state and chord configuration, where each `delta_j` started and where the MAP put
it, one panel per `N > 1` whose axis spans the prior box. The start is the run's own
prior draw, rebuilt from its seed; the bar is the 5 to 95% Laplace band; the data carry
`delta_j = 1` (dashed). A draw depends only on the seed and the volume, so a start sits
at the same `ln delta_j / ln N` at every `N` and in every configuration. Converged runs
only.

In [ ]:
"""start_rows = []
for record in converged_records:
    bound = record.settings.thomson_calibration_bound
    if bound <= 1.0:
        continue
    arrays = inference_store_utils.load_inference(
        inference_store_utils.INFERENCE_STORE, record.inference_id
    )[1]
    volumes = calibration.loc[
        (calibration["profile_id"] == record.truth_name)
        & (calibration["bound"] == bound)
        & (calibration["chords"] == chord_count(record)),
        "volume_index",
    ]
    # The reported MAP is the converged attempt of highest posterior, or the best
    # attempt when none converged, as `inference_run_utils` picks it.
    attempts_converged = arrays["start_converged"]
    candidates = (
        attempts_converged
        if attempts_converged.any()
        else np.ones_like(attempts_converged)
    )
    best = np.flatnonzero(candidates)[
        np.argmax(arrays["start_log_posterior"][candidates])
    ]
    # A start draw is keyed by the site name, so the calibration priors alone redraw
    # the run's own `delta_j` starts.
    start = inference_model_utils.start_points(
        record.settings, priors_utils.calibration_priors(volumes, bound)
    )[best]
    start_rows += [
        {
            "profile_id": record.truth_name,
            "bound": bound,
            "chords": chord_count(record),
            "volume_index": int(name.split("_")[-1]),
            "start": float(value),
        }
        for name, value in start.items()
    ]
corrections = calibration.merge(
    pd.DataFrame(start_rows), on=["profile_id", "bound", "chords", "volume_index"]
)

for (truth, count), truth_rows in corrections.groupby(["profile_id", "chords"]):
    figure, axes_row = pubfig.subplots(
        ncols=truth_rows["bound"].nunique(), width="double", aspect=1.6, sharex=True
    )
    for axes, (bound, rows) in zip(axes_row, truth_rows.groupby("bound")):
        radius = major_radius[rows["volume_index"]]
        axes.axhline(1.0, color="0.5", lw=0.8, ls="--", label=r"truth, $\delta_j = 1$")
        axes.errorbar(
            radius,
            rows["map"],
            yerr=[rows["map"] - rows["q05"], rows["q95"] - rows["map"]],
            fmt="o",
            ms=3.0,
            lw=1.0,
            color=pubfig.PALETTE[0],
            label="MAP, 5 to 95%",
        )
        axes.plot(
            radius,
            rows["start"],
            ls="none",
            marker="o",
            ms=4.0,
            mfc="none",
            mew=0.9,
            color=pubfig.PALETTE[1],
            label="initial guess",
        )
        # The prior is uniform in ln delta_j, so a log axis over its box shows where
        # in the prior the start and the MAP sit.
        axes.set_yscale("log")
        axes.set_ylim(bound**-1.08, bound**1.08)
        axes.set_yticks(
            [1.0 / bound, 1.0, bound],
            labels=[f"{1.0 / bound:.3g}", "1", f"{bound:g}"],
        )
        axes.set_yticks([], minor=True)
        axes.set_title(rf"$f_\delta$ = {bound:g}")
    axes_row[0].set_ylabel(r"$\delta_j$")
    figure.supxlabel(r"$R$ of the scattering volume [m]")
    handles, labels = axes_row[0].get_legend_handles_labels()
    # Computer Modern text has no underscore glyph and sets `_` as a raised dot.
    figure.legend(
        handles,
        labels,
        loc="outside upper center",
        ncols=3,
        frameon=False,
        title=truth.replace("_", r"$\_$") + f", {count} chords",
    )
    pubfig.save(
        figure,
        str(FIGURES_DIRECTORY / f"ts_calibration_bound_corrections_{truth}_if{count}"),
    )
    display(figure)
    plt.close(figure)"""

## 5. Information gain on the profile parameters

The information gain of the placement optimization, `0.5 log2 det[P^-1]_I / det[(J^T J
+ P)^-1]_I` (thesis Eq. placement-information-gain), on the `ne` parameters and on the
`ne` and `Te` parameters, every other parameter, the `delta_j` included, integrated out.

- **Expected**: computed as the placement does, with its functions
  (`placement_posterior`, `truth_point`, `prior_precision`, the joint study's `thomson_rows`,
  the interferometer study's `chord_rows`, `information_gain_bits`): `J` and `P`
  linearized at each state's two-power fit with every `delta_j = 1`, before any data,
  the chords on their in-plasma segments of the reference equilibrium, and the mean
  over the eight states. `N` enters through the prior precision of the `delta_j`, a
  configuration through which chord rows enter `J^T J`.
- **Realized**: the same formula with `J^T J` and `P` evaluated at each run's MAP, read
  from its stored Laplace arrays, and the mean over the converged runs.

The two differ only in the point they are evaluated at. The figure shows the
expected gain, one panel per set of parameters of interest with its own colour
scale; the table below it lists both gains.

In [ ]:
INTEREST_PREFIXES = {
    r"$n_e(\rho)$": ("ne_",),
    r"$n_e(\rho)$ and $T_e(\rho)$": ("ne_", "te_"),
}

surrogate = model_storage_utils.load_surrogate(
    PACKAGE_DIRECTORY / "mhd_equilibrium_forward_models" / "trained_models",
    record.surrogate_name,
    DATA_DIRECTORY,
    machine_scale=reference.machine_scale,
)
geometry = placement.thomson_geometry
channels_shape = (
    len(geometry.volume_positions),
    len(placement.polychromator_response.filter_centers),
)
template = inference_target_types.ThomsonSignalTarget(
    geometry=geometry,
    response=placement.polychromator_response,
    signals=jnp.zeros(channels_shape),
    uncertainties=jnp.ones(channels_shape),
    global_calibration=thomson_scattering_utils.absolute_calibration(geometry),
)
layout = {
    name: jnp.asarray(getattr(geometry, name))
    for name in joint_placement_study.LAYOUT_FIELDS
}
layout["band_scales"] = jnp.asarray(placement.polychromator_response.band_scales)
layout["global_calibration"] = template.global_calibration
# The placement scores a chord on its in-plasma segment of the reference equilibrium.
segments = geometry_utils.trim_chords_to_plasma(
    placement.interferometer_chords, reference.coefficients
)
starts = jnp.stack([jnp.asarray(chord.chord_start) for chord in segments])
ends = jnp.stack([jnp.asarray(chord.chord_end) for chord in segments])
truth_states = {
    truth: desc_truth_types.read_desc_truth(truth, PARQUET, SETTINGS)
    for truth in truths
}

gain_rows = []
for level in levels:
    settings = next(
        r.settings for r in records if r.settings.thomson_calibration_bound == level
    )
    posterior, priors = diagnostics_placement_optimization_utils.placement_posterior(
        settings, surrogate, template
    )
    interest = {
        name: diagnostics_placement_optimization_utils.parameter_indices(
            posterior, prefixes
        )
        for name, prefixes in INTEREST_PREFIXES.items()
    }
    thomson_rows_of = jax.jit(
        functools.partial(
            joint_placement_study.thomson_rows,
            template=template,
            posterior=posterior,
            settings=settings,
            surrogate=surrogate,
            signal_scale=placement.thomson_signal_scale,
            noise_model=placement.design.thomson_noise_model,
        )
    )
    chord_rows_of = jax.jit(
        functools.partial(
            interferometer_placement_study.chord_rows,
            posterior=posterior,
            settings=settings,
            surrogate=surrogate,
        )
    )
    for truth in truths:
        point = diagnostics_placement_optimization_utils.truth_point(
            truth_states[truth], posterior
        )
        precision = laplace_approximation_utils.prior_precision(
            point, posterior, priors
        )
        thomson = thomson_rows_of(point, layout)
        chords = chord_rows_of(point, starts, ends)
        for count in chord_counts:
            read = chords[np.asarray(indices_of[count], dtype=int)]
            information = thomson.T @ thomson + read.T @ read
            for name, indices in interest.items():
                gain_rows.append(
                    {
                        "gain": "expected",
                        "interest": name,
                        "chords": count,
                        "N": level,
                        "bits": float(
                            diagnostics_placement_optimization_utils.information_gain_bits(
                                information, precision, indices
                            )
                        ),
                    }
                )
    for run in converged_records:
        if run.settings.thomson_calibration_bound != level:
            continue
        arrays = inference_store_utils.load_inference(
            inference_store_utils.INFERENCE_STORE, run.inference_id
        )[1]
        # The stored Laplace arrays follow the site order of the same flat model.
        assert [str(name) for name in arrays["trajectory_site_names"]] == list(
            posterior.site_names
        )
        for name, indices in interest.items():
            gain_rows.append(
                {
                    "gain": "realized",
                    "interest": name,
                    "chords": chord_count(run),
                    "N": level,
                    "bits": float(
                        diagnostics_placement_optimization_utils.information_gain_bits(
                            jnp.asarray(arrays["laplace_information"]),
                            jnp.asarray(arrays["laplace_prior_precision"]),
                            indices,
                        )
                    ),
                }
            )
gains = pd.DataFrame(gain_rows)

colormap = plt.get_cmap("viridis").copy()
colormap.set_bad("0.9")
# The figure shows the expected gain, the placement objective; the table below lists
# the realized gain beside it.
figure, axes_row = pubfig.subplots(1, 2, width="double", aspect=0.7)
for axes, interest_name in zip(axes_row, INTEREST_PREFIXES):
    values = np.asarray(
        gains[(gains["gain"] == "expected") & (gains["interest"] == interest_name)]
        .pivot_table(index="chords", columns="N", values="bits")
        .reindex(index=chord_counts, columns=levels),
        dtype=float,
    )
    # One colour scale per panel, since the two gains span different ranges.
    low, high = np.nanmin(values), np.nanmax(values)
    midpoint = 0.5 * (low + high)
    n_rows, n_columns = values.shape
    mesh = axes.pcolormesh(
        np.arange(n_columns + 1) - 0.5,
        np.arange(n_rows + 1) - 0.5,
        np.ma.masked_invalid(values),
        cmap=colormap,
        vmin=low,
        vmax=high,
    )
    for row in range(n_rows):
        for col in range(n_columns):
            if np.isfinite(values[row, col]):
                axes.text(
                    col,
                    row,
                    f"{values[row, col]:.1f}",
                    ha="center",
                    va="center",
                    fontsize="x-small",
                    color="w" if values[row, col] < midpoint else "k",
                )
    axes.set_xticks(np.arange(n_columns), [f"{level:g}" for level in levels])
    axes.set_yticks(np.arange(n_rows), [str(count) for count in chord_counts])
    axes.set_xlabel(r"$f_\delta$")
    figure.colorbar(mesh, ax=axes, label=f"{interest_name} EIG [bits]", shrink=0.9)
axes_row[0].set_ylabel("interferometer chords")
pubfig.save(figure, str(FIGURES_DIRECTORY / "ts_calibration_bound_information_gain"))
display(figure)
plt.close(figure)

gains.pivot_table(
    index=["interest", "gain", "chords"], columns="N", values="bits"
).round(1)

## 6. Summary

Per state, level and chord configuration, for `ne` and `Te` in separate tables: the core
error at `rho = 0` (% of the true value), the RMSE over the radius (% of the true peak)
and the median band half-width (% of the local value), over the converged runs. The last
two rows are the mean and the standard deviation over the states.

In [ ]:
summary = metrics.pivot_table(
    index="truth_name",
    columns=["chords", "profile", "N"],
    values=["axis_error_percent", "error_percent", "uncertainty_percent"],
    aggfunc="mean",
).rename(
    columns={
        "axis_error_percent": "core error [%]",
        "error_percent": "RMSE [%]",
        "uncertainty_percent": "band half-width [%]",
    }
)
summary = summary.rename_axis(
    index="truth", columns=["metric", "chords", "profile", "N"]
)
summary = pd.concat([summary, summary.agg(["mean", "std"])])
for count in chord_counts:
    for profile in ("ne", "Te"):
        table = summary.xs((count, profile), level=("chords", "profile"), axis=1)[
            ["core error [%]", "RMSE [%]", "band half-width [%]"]
        ]
        print(f"{profile}, {count} chords")
        display(table.round(2))